In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# ============================================
# HAFTA 4 — Data Quality Framework
# Real World Case: Bozuk veri dashboard'a yansıdı!
# Çözüm: Silver tablosuna girmeden önce kontrol et
# ============================================

# Silver tablosunu oku
df = spark.read.table("silver_sales")

print("=" * 50)
print("📊 SILVER_SALES VERİ KALİTE RAPORU")
print("=" * 50)

# 1. Temel bilgiler
print(f"\n📌 Toplam kayıt sayısı: {df.count()}")
print(f"📌 Kolon sayısı: {len(df.columns)}")
print(f"📌 Kolonlar: {df.columns}")

# 2. NULL kontrolü
print("\n🔍 NULL DEĞER KONTROLÜ:")
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    *[spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns])

null_counts.show()

# 3. Temel istatistikler
print("\n📈 TEMEL İSTATİSTİKLER:")
df.select("SALES", "QUANTITYORDERED", "PRICEEACH").describe().show()

StatementMeta(, bd156288-d361-46e0-9c5a-58a5b30822dc, 3, Finished, Available, Finished, False)

📊 SILVER_SALES VERİ KALİTE RAPORU

📌 Toplam kayıt sayısı: 2823
📌 Kolon sayısı: 25
📌 Kolonlar: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

🔍 NULL DEĞER KONTROLÜ:
+-----------+---------------+---------+---------------+-----+---------+------+------+--------+-------+-----------+----+-----------+------------+-----+------------+------------+----+-----+----------+-------+---------+---------------+----------------+--------+
|ORDERNUMBER|QUANTITYORDERED|PRICEEACH|ORDERLINENUMBER|SALES|ORDERDATE|STATUS|QTR_ID|MONTH_ID|YEAR_ID|PRODUCTLINE|MSRP|PRODUCTCODE|CUSTOMERNAME|PHONE|ADDRESSLINE1|ADDRESSLINE2|CITY|STATE|POSTALCODE|COUNTRY|TERRITORY|CONTACTLASTNAME|CONTACTFIRSTNAME|DEALSIZE|
+-----------+---------------+--

In [2]:
# ============================================
# VERİ KALİTE KURALLARI
# Kritik kolonlar NULL olamaz
# SALES negatif olamaz
# QUANTITYORDERED 0'dan büyük olmalı
# ============================================

from pyspark.sql.functions import col

print("=" * 50)
print("🛡️ VERİ KALİTE KURAL KONTROLLERI")
print("=" * 50)

# KURAL 1 — Kritik kolonlarda NULL kontrolü
print("\n📌 KURAL 1: Kritik kolon NULL kontrolü")
critical_nulls = df.filter(
    col("SALES").isNull() |
    col("ORDERNUMBER").isNull() |
    col("CUSTOMERNAME").isNull() |
    col("ORDERDATE").isNull() |
    col("COUNTRY").isNull()
).count()

if critical_nulls == 0:
    print(f"✅ PASSED — Kritik kolonlarda NULL yok")
else:
    print(f"❌ FAILED — {critical_nulls} kayıtta kritik NULL var!")

# KURAL 2 — SALES negatif olamaz
print("\n📌 KURAL 2: Negatif SALES kontrolü")
negative_sales = df.filter(col("SALES") < 0).count()

if negative_sales == 0:
    print(f"✅ PASSED — Negatif SALES yok")
else:
    print(f"❌ FAILED — {negative_sales} kayıtta negatif SALES var!")

# KURAL 3 — QUANTITYORDERED 0'dan büyük olmalı
print("\n📌 KURAL 3: QUANTITYORDERED > 0 kontrolü")
zero_qty = df.filter(col("QUANTITYORDERED") <= 0).count()

if zero_qty == 0:
    print(f"✅ PASSED — Tüm siparişlerde quantity > 0")
else:
    print(f"❌ FAILED — {zero_qty} kayıtta geçersiz quantity var!")

# KURAL 4 — PRICEEACH mantıklı aralıkta mı?
print("\n📌 KURAL 4: PRICEEACH aralık kontrolü (0-1000)")
invalid_price = df.filter(
    (col("PRICEEACH") <= 0) | # --or (veya)
    (col("PRICEEACH") > 1000)
).count()

if invalid_price == 0:
    print(f"✅ PASSED — Tüm fiyatlar geçerli aralıkta")
else:
    print(f"❌ FAILED — {invalid_price} kayıtta geçersiz fiyat var!")

# KURAL 5 — COUNTRY boş olamaz
print("\n📌 KURAL 5: COUNTRY uniqueness kontrolü")
country_count = df.select("COUNTRY").distinct().count()
print(f"✅ Toplam unique ülke sayısı: {country_count}")

print("\n" + "=" * 50)
print("🎯 VERİ KALİTE KONTROLÜ TAMAMLANDI")
print("=" * 50)

StatementMeta(, bd156288-d361-46e0-9c5a-58a5b30822dc, 4, Finished, Available, Finished, False)

🛡️ VERİ KALİTE KURAL KONTROLLERI

📌 KURAL 1: Kritik kolon NULL kontrolü
✅ PASSED — Kritik kolonlarda NULL yok

📌 KURAL 2: Negatif SALES kontrolü
✅ PASSED — Negatif SALES yok

📌 KURAL 3: QUANTITYORDERED > 0 kontrolü
✅ PASSED — Tüm siparişlerde quantity > 0

📌 KURAL 4: PRICEEACH aralık kontrolü (0-1000)
✅ PASSED — Tüm fiyatlar geçerli aralıkta

📌 KURAL 5: COUNTRY uniqueness kontrolü
✅ Toplam unique ülke sayısı: 19

🎯 VERİ KALİTE KONTROLÜ TAMAMLANDI


In [3]:
from pyspark.sql import Row
from datetime import datetime

# Otomatik — önceki değişkenlerden alıyor!
quality_results = [
    Row(
        check_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        table_name="silver_sales",
        rule_name="Kritik NULL Kontrolü",
        status="PASSED" if critical_nulls == 0 else "FAILED",
        failed_count=critical_nulls
    ),
    Row(
        check_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        table_name="silver_sales",
        rule_name="Negatif SALES Kontrolü",
        status="PASSED" if negative_sales == 0 else "FAILED",
        failed_count=negative_sales
    ),
    Row(
        check_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        table_name="silver_sales",
        rule_name="QUANTITYORDERED > 0",
        status="PASSED" if zero_qty == 0 else "FAILED",
        failed_count=zero_qty
    ),
    Row(
        check_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        table_name="silver_sales",
        rule_name="PRICEEACH Aralık Kontrolü",
        status="PASSED" if invalid_price == 0 else "FAILED",
        failed_count=invalid_price
    ),
    Row(
        check_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
        table_name="silver_sales",
        rule_name="COUNTRY Uniqueness",
        status="PASSED" if country_count == 19 else "FAILED",
        failed_count=0 if country_count == 19 else country_count
    ),
]

# DataFrame'e çevir ve kaydet
quality_df = spark.createDataFrame(quality_results)

quality_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_data_quality_log")

print("✅ Veri kalite logu otomatik kaydedildi!")
display(quality_df)

StatementMeta(, bd156288-d361-46e0-9c5a-58a5b30822dc, 5, Finished, Available, Finished, False)

✅ Veri kalite logu otomatik kaydedildi!


SynapseWidget(Synapse.DataFrame, 67f3051c-3bec-49a2-a481-7ad4f8d73526)

In [4]:
# ============================================
# FAILED DURUMUNDA AKSİYON
# Kritik hata varsa pipeline durduruluyor
# Düzeltilebilir hata varsa otomatik düzeltiliyor
# ============================================

print("=" * 50)
print("🔧 OTOMATİK DÜZELTME & AKSİYON KATMANI")
print("=" * 50)

# ---- DÜZELTİLEBİLİR HATALAR ----
from pyspark.sql.functions import col, when, lit

# DÜZELTME 1: ADDRESSLINE2 NULL → "N/A" ile doldur
# (ADDRESSLINE1 zaten tamamen dolu, düzeltmeye gerek yok)
df_fixed = df.withColumn(
    "ADDRESSLINE2",
    when(col("ADDRESSLINE2").isNull(), lit("N/A"))
    .otherwise(col("ADDRESSLINE2"))
)

# DÜZELTME 2: STATE NULL → "N/A" ile doldur
df_fixed = df_fixed.withColumn(
    "STATE",
    when(col("STATE").isNull(), lit("N/A"))
    .otherwise(col("STATE"))
)

# DÜZELTME 3: POSTALCODE NULL → "00000" ile doldur
df_fixed = df_fixed.withColumn(
    "POSTALCODE",
    when(col("POSTALCODE").isNull(), lit("00000"))
    .otherwise(col("POSTALCODE"))
)

# Kaç kayıt düzeltildi?
fixed_address2 = df.filter(col("ADDRESSLINE2").isNull()).count()
fixed_state = df.filter(col("STATE").isNull()).count()
fixed_postal = df.filter(col("POSTALCODE").isNull()).count()

print(f"\n✅ DÜZELTMELER:")
print(f"   ADDRESSLINE2: {fixed_address2} kayıt → 'N/A' ile dolduruldu")
print(f"   STATE: {fixed_state} kayıt → 'N/A' ile dolduruldu")
print(f"   POSTALCODE: {fixed_postal} kayıt → '00000' ile dolduruldu")

# Düzeltilmiş veriyi Silver'a geri kaydet
df_fixed.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_sales")

print(f"\n✅ Düzeltilmiş veri silver_sales'e kaydedildi!")

# ---- KRİTİK HATALAR — PIPELINE DURDUR ----
print("\n" + "=" * 50)
print("🚨 KRİTİK HATA KONTROLÜ")
print("=" * 50)

# KONTROL 1: Kritik kolonlarda NULL
if critical_nulls > 0:
    raise Exception(
        f"KRİTİK HATA: {critical_nulls} kayıtta kritik NULL var! "
        f"Pipeline durduruldu. Veriyi kontrol et!"
    )

# KONTROL 2: Negatif SALES
if negative_sales > 0:
    raise Exception(
        f"KRİTİK HATA: {negative_sales} kayıtta negatif SALES var! "
        f"Pipeline durduruldu. Veriyi kontrol et!"
    )

# KONTROL 3: Geçersiz QUANTITY
if zero_qty > 0:
    raise Exception(
        f"KRİTİK HATA: {zero_qty} kayıtta geçersiz quantity var! "
        f"Pipeline durduruldu. Veriyi kontrol et!"
    )

# KONTROL 4: Geçersiz fiyat
if invalid_price > 0:
    raise Exception(
        f"KRİTİK HATA: {invalid_price} kayıtta geçersiz fiyat var! "
        f"Pipeline durduruldu. Veriyi kontrol et!"
    )

# KONTROL 5: COUNTRY uniqueness
# Hiç ülke yoksa veya beklenmedik sayıda ülke varsa durdur
if country_count == 0:
    raise Exception(
        f"KRİTİK HATA: Hiç ülke verisi yok! "
        f"Pipeline durduruldu!"
    )

if country_count > 50:
    raise Exception(
        f"KRİTİK HATA: Beklenmedik ülke sayısı: {country_count}! "
        f"Veri kalitesi sorunu olabilir. Pipeline durduruldu!"
    )

print("✅ Tüm kritik kontroller geçti!")
print("✅ Pipeline devam edebilir! 🚀")
print("=" * 50)

StatementMeta(, bd156288-d361-46e0-9c5a-58a5b30822dc, 6, Finished, Available, Finished, False)

🔧 OTOMATİK DÜZELTME & AKSİYON KATMANI

✅ DÜZELTMELER:
   ADDRESSLINE2: 0 kayıt → 'N/A' ile dolduruldu
   STATE: 0 kayıt → 'N/A' ile dolduruldu
   POSTALCODE: 0 kayıt → '00000' ile dolduruldu

✅ Düzeltilmiş veri silver_sales'e kaydedildi!

🚨 KRİTİK HATA KONTROLÜ
✅ Tüm kritik kontroller geçti!
✅ Pipeline devam edebilir! 🚀


In [5]:
# ============================================
# HAFTA 4 GÜN 2 — UNIT TESTING
# Real World Case: Kod değişikliği SCD mantığını bozdu!
# Çözüm: Otomatik testler ile yakala
# ============================================

print("=" * 50)
print("🧪 UNIT TEST SUITE — RetailMart360")
print("=" * 50)

# Test sonuçlarını takip etmek için
test_results = []

def run_test(test_name, condition, error_message):
    """
    Her test için:
    - condition True ise → PASSED
    - condition False ise → FAILED + hata mesajı
    """
    if condition:
        print(f"✅ PASSED: {test_name}")
        test_results.append({"test": test_name, "status": "PASSED"})
    else:
        print(f"❌ FAILED: {test_name} → {error_message}")
        test_results.append({"test": test_name, "status": "FAILED"})

print("\n📌 SILVER_SALES TESTLERİ:")

# TEST 1: Tablo boş olmamalı
run_test(
    "silver_sales boş değil",
    df.count() > 0,
    "silver_sales tablosu boş!"
)

# TEST 2: Beklenen kolon sayısı
run_test(
    "Kolon sayısı doğru (25 kolon)",
    len(df.columns) == 25,
    f"Beklenen 25 kolon, bulunan: {len(df.columns)}"
)

# TEST 3: Kritik kolonlar mevcut
critical_columns = ["ORDERNUMBER", "SALES", "CUSTOMERNAME", 
                    "ORDERDATE", "COUNTRY"]
columns_exist = all(c in df.columns for c in critical_columns)
run_test(
    "Kritik kolonlar mevcut",
    columns_exist,
    f"Eksik kritik kolon var!"
)

# TEST 4: SALES pozitif olmalı
run_test(
    "Tüm SALES değerleri pozitif",
    df.filter(col("SALES") <= 0).count() == 0,
    "Negatif veya sıfır SALES değeri var!"
)

# TEST 5: NULL düzeltmeleri çalıştı mı?
run_test(
    "ADDRESSLINE2 NULL kalmadı",
    df_fixed.filter(col("ADDRESSLINE2").isNull()).count() == 0,
    "ADDRESSLINE2'de hâlâ NULL var!"
)

run_test(
    "STATE NULL kalmadı",
    df_fixed.filter(col("STATE").isNull()).count() == 0,
    "STATE'de hâlâ NULL var!"
)

run_test(
    "POSTALCODE NULL kalmadı",
    df_fixed.filter(col("POSTALCODE").isNull()).count() == 0,
    "POSTALCODE'da hâlâ NULL var!"
)

print("\n📌 DIM_CUSTOMER_SCD2 TESTLERİ:")

dim_customer = spark.read.table("dim_customer_scd2")

# TEST 6: SCD2 tablosu boş olmamalı
run_test(
    "dim_customer_scd2 boş değil",
    dim_customer.count() > 0,
    "dim_customer_scd2 tablosu boş!"
)

# TEST 7: is_current kolonu var mı?
run_test(
    "is_current kolonu mevcut",
    "is_current" in dim_customer.columns,
    "is_current kolonu eksik — SCD2 bozulmuş olabilir!"
)

# TEST 8: Her müşteri için en fazla 1 aktif kayıt olmalı
from pyspark.sql.functions import sum as spark_sum
duplicate_current = dim_customer\
    .filter(col("is_current") == True)\
    .groupBy("CUSTOMERNAME")\
    .count()\
    .filter(col("count") > 1)\
    .count()

run_test(
    "Her müşteri için tek aktif SCD2 kaydı var",
    duplicate_current == 0,
    f"{duplicate_current} müşteride birden fazla aktif kayıt var!"
)

print("\n📌 DIM_GEOGRAPHY_COUNTRY TESTLERİ:")

dim_geo = spark.read.table("dim_geography_country")

# TEST 9: 19 unique ülke olmalı
run_test(
    "dim_geography_country 19 ülke içeriyor",
    dim_geo.count() == 19,
    f"Beklenen 19 ülke, bulunan: {dim_geo.count()}"
)

# TEST 10: Singapore APAC'ta olmalı

from pyspark.sql.functions import upper

singapore_territory = dim_geo\
    .filter(upper(col("COUNTRY")) == "SINGAPORE")\
    .select("TERRITORY")\
    .collect()

run_test(
    "Singapore APAC territory'sinde",
    len(singapore_territory) > 0 and 
    singapore_territory[0]["TERRITORY"].upper() == "APAC",
    "Singapore'un territory'si APAC değil!"
)

# ---- TEST ÖZETI ----
print("\n" + "=" * 50)
print("📊 TEST ÖZETI:")

passed = sum([1 for t in test_results if t["status"] == "PASSED"])
failed = sum([1 for t in test_results if t["status"] == "FAILED"])

total = len(test_results)

print(f"✅ Passed: {passed}/{total}")
print(f"❌ Failed: {failed}/{total}")

if failed > 0:
    raise Exception(
        f"❌ {failed} TEST BAŞARISIZ! "
        f"Pipeline durduruldu. Testleri kontrol et!"
    )
else:
    print("\n🎉 TÜM TESTLER BAŞARILI — Pipeline devam edebilir!")
print("=" * 50)

StatementMeta(, bd156288-d361-46e0-9c5a-58a5b30822dc, 7, Finished, Available, Finished, False)

🧪 UNIT TEST SUITE — RetailMart360

📌 SILVER_SALES TESTLERİ:
✅ PASSED: silver_sales boş değil
✅ PASSED: Kolon sayısı doğru (25 kolon)
✅ PASSED: Kritik kolonlar mevcut
✅ PASSED: Tüm SALES değerleri pozitif
✅ PASSED: ADDRESSLINE2 NULL kalmadı
✅ PASSED: STATE NULL kalmadı
✅ PASSED: POSTALCODE NULL kalmadı

📌 DIM_CUSTOMER_SCD2 TESTLERİ:
✅ PASSED: dim_customer_scd2 boş değil
✅ PASSED: is_current kolonu mevcut
✅ PASSED: Her müşteri için tek aktif SCD2 kaydı var

📌 DIM_GEOGRAPHY_COUNTRY TESTLERİ:
✅ PASSED: dim_geography_country 19 ülke içeriyor
✅ PASSED: Singapore APAC territory'sinde

📊 TEST ÖZETI:
✅ Passed: 12/12
❌ Failed: 0/12

🎉 TÜM TESTLER BAŞARILI — Pipeline devam edebilir!


In [1]:
# ============================================
# HAFTA 4 GÜN 4 — Incremental Load & Watermark
# Real World Case: Her gün milyonlarca satır var
# Sadece yeni verileri işle!
# ============================================

from pyspark.sql.functions import max, lit
from datetime import datetime

print("=" * 50)
print("⚡ INCREMENTAL LOAD BAŞLIYOR")
print("=" * 50)

# 1. Watermark tablosu var mı kontrol et
# Yoksa oluştur
try:
    watermark_df = spark.read.table("watermark_log")
    last_watermark = watermark_df\
        .orderBy("load_date", ascending=False)\
        .limit(1)\
        .collect()[0]["watermark_date"]
    print(f"📌 Son watermark: {last_watermark}")
except:
    # İlk kez çalışıyor — başlangıç tarihi olarak en eski tarihi al
    last_watermark = "2000-01-01"
    print(f"📌 İlk çalışma — watermark: {last_watermark}")

# 2. Sadece yeni kayıtları getir
new_data = spark.read.table("silver_sales")\
    .filter(f"ORDERDATE > '{last_watermark}'")

new_record_count = new_data.count()
print(f"✅ Yeni kayıt sayısı: {new_record_count}")

# 3. Yeni watermark belirle
if new_record_count > 0:
    new_watermark = new_data\
        .select(max("ORDERDATE"))\
        .collect()[0][0]
    print(f"✅ Yeni watermark: {new_watermark}")
else:
    new_watermark = last_watermark
    print("ℹ️ Yeni kayıt yok — watermark değişmedi")

# 4. Watermark'ı kaydet
from pyspark.sql import Row

watermark_row = spark.createDataFrame([
    Row(
        load_date=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        watermark_date=str(new_watermark),
        new_records=new_record_count
    )
])

watermark_row.write.format("delta")\
    .mode("append")\
    .saveAsTable("watermark_log")

print(f"\n✅ Watermark kaydedildi!")
print("=" * 50)

StatementMeta(, 81135dc3-ec59-466f-855f-8272d39f58f2, 3, Finished, Available, Finished, False)

⚡ INCREMENTAL LOAD BAŞLIYOR
📌 İlk çalışma — watermark: 2000-01-01
✅ Yeni kayıt sayısı: 2823
✅ Yeni watermark: 2005-05-31

✅ Watermark kaydedildi!
